In [53]:
import os
import pandas as pd
import sys
import re
from pathlib import Path
from dotenv import load_dotenv
import faiss
import numpy as np
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

True

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks" and ROOT.parent.name == "src":
    ROOT = ROOT.parent.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PATH = ROOT / "data" / "evenements-publics-openagenda.csv"

In [59]:
raw = pd.read_csv(DATA_PATH, sep=";")
raw.head()

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,25926303,venez-rencontrer-yes-industrie-et-logistique-p...,https://openagenda.com/francetravail/events/ve...,Venez rencontrer YES (industrie et logistique)...,Présence de l'agence d'emploi YES de 11h à 12h...,<p>Présence de l'agence d'emploi YES de 11h à ...,NaN,"INTERIM,#TousMobilisés,Marché du travail,Recru...",https://cdn.openagenda.com/main/05c4bea3d69946...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
1,9825858,atelier-decouverte-du-numerique-8424261,https://openagenda.com/francetravail/events/at...,ATELIER DECOUVERTE DU NUMERIQUE,Vous souhaitez vous orienter et travailler dan...,<p>Vous souhaitez vous orienter et travailler ...,NaN,"L alternance,Métiers du numérique et informati...",https://cdn.openagenda.com/main/0ef2e72ec05543...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
2,30636112,rencontrez-synergie-proxi-latecoere-logistique...,https://openagenda.com/francetravail/events/re...,Rencontrez SYNERGIE PROXI LATECOERE (logistiqu...,Présence de l'agence Synergie Proxi Latécoère ...,<p>Présence de l'agence Synergie Proxi Latécoè...,NaN,"1 jeune 1 solution,#TousMobilisés,Recrutement",https://cdn.openagenda.com/main/f710e417553841...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
3,38746001,handicap-et-emploi-les-cles-de-ma-reussite-res...,https://openagenda.com/francetravail/events/ha...,Handicap et emploi : les clés de ma réussite (...,Objectifs : vous apporter les éléments clés qu...,<p>Objectifs : vous apporter les éléments clés...,NaN,"Se préparer,Marché du travail",https://cdn.openagenda.com/main/7faa84049d9640...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN
4,85266313,eiffage-energie-systemes-formation-metier-mont...,https://openagenda.com/francetravail/events/ei...,EIFFAGE ENERGIE SYSTEMES : Formation métier Mo...,Vous cherchez à vous orienter vers un métier q...,<p>Vous cherchez à vous orienter vers un métie...,NaN,"#TousMobilisés,Recrutement,Découverte formatio...",https://cdn.openagenda.com/main/d097609d3acd4f...,NaN,...,38495884.0,NaN,NaN,NaN,NaN,NaN,NaN,France (Métropole),"[{""type"": ""link"", ""value"": ""https://meseveneme...",NaN


### Data Cleaning

##### Remove duplicates

In [4]:
raw.description_fr.describe()

count                                                  4904
unique                                                 3048
top       Le Job Dating des Minimes se déroule le mardi ...
freq                                                     93
Name: description_fr, dtype: object

Il y n'y a que 3048 valeurs uniques dans la colonne descrition_fr du tableau de donnée. Traiter les mêmes éléments avec le modèle d'embedding textuel créerai les mêmes embeddings, on aurait des valeurs en doubles dans la base de données, ce n'est pas utile. Je ne vais donc conserver que les valeurs uniques pour la création de la table d'embeddings.

In [ ]:
unique_description_fr = raw.description_fr.unique()

#### Remove missing values

In [ ]:

unique_description_fr.isna().sum()

np.int64(1)

unique_description_fr contient une valeur nulle qu'on va pouvoir supprimer avec la fonction dropna

In [6]:
unique_description_fr = unique_description_fr.dropna()

#### Remove dry emails, urls or numerics

In [ ]:
description_df = pd.DataFrame(unique_description_fr, columns = ['text'])
URL_PATTERN = re.compile(r'^https?://\S+$', re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'^[\w\.-]+@[\w\.-]+\.\w+$')

def check_for_dry_emails_or_urls(text_input):

    text = text_input.strip()

    if URL_PATTERN.fullmatch(text):
        return {"valid": False, "reason": "url_only"}

    if EMAIL_PATTERN.fullmatch(text):
        return {"valid": False, "reason": "email_only"}

    if text.isnumeric():
        return {"valid": False, "reason": "numeric_only"}

    return {
        "valid": True,
        "reason": None
    }

In [ ]:
quality = description_df["text"].apply(check_for_dry_emails_or_urls).apply(pd.Series)

df_quality = pd.concat(
    [description_df, quality],
    axis=1
)

In [12]:
df_quality["reason"].value_counts(dropna=False)

reason
NaN         3047
url_only       1
Name: count, dtype: int64

Il n'y a qu'un seul élément avec une forme problématique, une description composée d'un url seul. Il n'est pas pertinent de le conserver dans la base de données, parce qu'il n'apportera pas d'information, on peut donc supprimer l'entrée.

In [ ]:
index_to_drop = df_quality[df_quality["reason"] == "url_only"].index
description_df = description_df.drop(index_to_drop).reset_index(drop=True)
description_df


,text
0,Présence de l'agence d'emploi YES de 11h à 12h...
1,Vous souhaitez vous orienter et travailler dan...
2,Présence de l'agence Synergie Proxi Latécoère ...
3,Objectifs : vous apporter les éléments clés qu...
4,Vous cherchez à vous orienter vers un métier q...
...,...
3042,Une sélection de films inédits ou présentés en...
3044,"Le jeudi 4 juin 2026, Le Cratère se met au dia..."
3045,Rétrospective Anja Breien Du 15 au 28 juillet ...
3046,"Rencontre avec Laura André-Boyet, instructrice..."


#### Check for input lenght

In [ ]:
# Je regarde d'abord le nombre de mots dans les descriptions
description_df["word_count"] = description_df["text"].str.split().str.len()

print(description_df["word_count"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))

count    3047.000000
mean       16.412537
std        10.342881
min         1.000000
1%          1.000000
5%          3.000000
25%         8.000000
50%        14.000000
75%        26.000000
95%        33.000000
99%        36.000000
max        46.000000
Name: word_count, dtype: float64


On peut voir qu'il y a un certain nombre de description très courtes, avant de les supprimer, il peut être intéressant de regarder leur pertinence.

In [ ]:
pd.options.display.max_rows = 4000
description_df[description_df["word_count"] < 4]

,text,char_length,word_count
42,Atelier d'écriture,18,2
49,"Elisabeth BRAUß, Piano",22,3
54,Rencontre,9,1
61,Festival Occitania 2026,23,3
96,EXplorations,12,1
103,Échanges,8,1
129,PE IRSD,7,2
130,Autour du clavecin,18,3
131,Grande Scène,12,2
133,Petites scènes,14,2


On observe que malgré la petite taille de ces descriptions, celles-ci semblent informative. En revanche il y a des variations de casse et des caractères spéciaux. Une étape de normalisation pourrait être utile.

In [31]:
description_df["char_length"] = description_df["text"].str.len()
print(description_df["char_length"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))


count    3047.000000
mean      103.808664
std        64.766182
min         3.000000
1%          8.000000
5%         17.000000
25%        47.000000
50%        90.000000
75%       170.000000
95%       200.000000
99%       200.000000
max       200.000000
Name: char_length, dtype: float64


In [32]:
description_df[description_df["char_length"] < 9]

,text,char_length,word_count
103,Échanges,8,1
129,PE IRSD,7,2
233,Piano,5,1
275,Balade,6,1
301,Fifigrot,8,1
362,Lecture,7,1
403,De Piano,8,2
521,Lectures,8,1
522,Atelier,7,1
525,Jeux,4,1


In [43]:
def normalize_text(text):
    text = text.lower() # Convertit tous les caractères en minuscules.
    text = re.sub(r'\s+', ' ', text) # Remplace les espaces multiples par un seul espace.
    text = re.sub(r"(?<=[a-zà-ÿ])-(?=[a-zà-ÿ])", " ", text) # Remplace les tirets par des espaces
    text = re.sub(r"[^a-z0-9à-ÿ\s']", "", text) # Supprime tous les caractères non alphanumériques.
    if re.match(r"^de\s+\w+$", text):
        text = re.sub(r"^de\s+", "", text) # Supprime le mot "de" au début de la description si présent (ex "de piano")
    return text.strip() # Supprime les espaces en trop au début et à la fin.

description_df["normalized_text"] = description_df["text"].apply(normalize_text)

In [44]:
description_df

,text,char_length,word_count,normalized_text
0,Présence de l'agence d'emploi YES de 11h à 12h...,200,33,présence de l'agence d'emploi yes de 11h à 12h...
1,Vous souhaitez vous orienter et travailler dan...,124,21,vous souhaitez vous orienter et travailler dan...
2,Présence de l'agence Synergie Proxi Latécoère ...,200,31,présence de l'agence synergie proxi latécoère ...
3,Objectifs : vous apporter les éléments clés qu...,108,17,objectifs vous apporter les éléments clés qui...
4,Vous cherchez à vous orienter vers un métier q...,58,11,vous cherchez à vous orienter vers un métier q...
5,Le Job Dating des Minimes se déroule le mardi ...,138,25,le job dating des minimes se déroule le mardi ...
6,Vous êtes intéressé(e) par les métiers de l'im...,179,25,vous êtes intéressée par les métiers de l'immo...
7,Le Job Dating des Minimes se déroule le mardi ...,196,34,le job dating des minimes se déroule le mardi ...
8,"Vous cherchez un emploi, une reconversion ou s...",124,21,vous cherchez un emploi une reconversion ou si...
9,"France Travail Bellefontaine, dans le cadre de...",163,23,france travail bellefontaine dans le cadre des...


In [45]:
description_df.normalized_text.describe()

count            3047
unique           2991
top       table ronde
freq                4
Name: normalized_text, dtype: object

In [46]:
clean_data = description_df.normalized_text.drop_duplicates()
clean_data

0       présence de l'agence d'emploi yes de 11h à 12h...
1       vous souhaitez vous orienter et travailler dan...
2       présence de l'agence synergie proxi latécoère ...
3       objectifs  vous apporter les éléments clés qui...
4       vous cherchez à vous orienter vers un métier q...
5       le job dating des minimes se déroule le mardi ...
6       vous êtes intéressée par les métiers de l'immo...
7       le job dating des minimes se déroule le mardi ...
8       vous cherchez un emploi une reconversion ou si...
9       france travail bellefontaine dans le cadre des...
10      équilibre de vie et projet professionnel vous ...
11      vous avez entre 17 et 35 ans  vous êtes de nat...
12      sophrologie sonothérapie  emploi  un moment po...
13      vous rêvez de créer votre entreprise ou de vou...
14      participez à ce job dating et rencontrez des e...
15      vous souhaitez changer de voie professionnelle...
16      jobdating dédié uniquement à l'alternance ou d...
17      vous a

In [49]:
def split_text_into_chunks(text, chunk_size = 1000, chunk_overlap = 200):
    # Initialisation du splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Taille de chaque segment
        chunk_overlap=chunk_overlap # Chevauchement entre les segments
        )
    # Découpage du texte
    segments = text_splitter.split_text(text)
    return segments
    
def get_embeddings(texts, model="text-embedding-3-small"):
    client = OpenAI()
    response = client.embeddings.create(
        input=texts,
        model=model
    )
    return response.data[0].embedding

In [47]:
len(clean_data)

2991

In [55]:
clean_data.tolist()

["présence de l'agence d'emploi yes de 11h à 12h des missions sont à pourvoir dans les métiers suivants   technicien de maintenance industriel hf  opérateur productionconducteur de ligne hf  chau",
 "vous souhaitez vous orienter et travailler dans les métiers du numérique  n'attendez plus cet atelier est fait pour vous",
 "présence de l'agence synergie proxi latécoère logistique de 11h à 11h45 des missions longue durée sont à pourvoir dans les métiers suivants être mobile est un critère indispensable   préparateur",
 "objectifs  vous apporter les éléments clés qui vous serviront dans la réussite de votre recherche d'emploi",
 'vous cherchez à vous orienter vers un métier qui recrute',
 'le job dating des minimes se déroule le mardi 09 juin à la maison toulouse services minimes 4 place du marché aux cochons 31200 toulouse',
 "vous êtes intéressée par les métiers de l'immobilier venez découvrir ce secteur métiers marché du travail formations salaires dans le cadre d'un atelier de découve

In [56]:
documents = clean_data.tolist()

# Génération des embeddings pour chaque document
embeddings = np.array([get_embeddings(doc) for doc in documents])

# Initialisation de l'index Faiss
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Ajout des embeddings à l'index
index.add(embeddings)

# Sauvegarde de l'index sur le disque
faiss.write_index(index, "faiss_index.idx")

In [58]:
# Requête utilisateur
query = "Informations les cours de piano"

# Génération de l'embedding de la requête
query_embedding = get_embeddings(query)

# Recherche des k documents les plus similaires
k = 5
distances, indices = index.search(np.array([query_embedding]), k)

# Affichage des résultats
for idx in indices[0]:
    print(f"Document {idx}: {documents[idx]}")

Document 1883: classes de piano
Document 592: classe de piano
Document 1561: classe de piano complémentaire
Document 2793: accompagnement piano
Document 232: piano


In [50]:
embeddings_list = []
for text in clean_data:
    embedding = get_embeddings(text)
    embeddings_list.append(embedding)

print(f"Number of embeddings: {len(embeddings_list)}")

Number of embeddings: 2991


In [ ]:
pd.DataFrame(embeddings_list)

In [24]:
print(response.data[0].embedding)

NameError: name 'response' is not defined

In [7]:


# 1. Cette ligne lit le fichier .env et charge la clé en mémoire
load_dotenv()

# 2. Le client OpenAI cherche automatiquement la variable "OPENAI_API_KEY"
client = OpenAI()

# 3. Vous pouvez maintenant utiliser l'API normalement
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "Explique-moi la lune en une phrase."}
    ]
)

print(completion.choices[0].message.content)


La Lune est le satellite naturel de la Terre, éclairée par le Soleil, influençant les marées et fascinant l'humanité par ses phases changeantes et son impact culturel et scientifique.


In [6]:
print(completion.choices[0].message.content)

La lune est le satellite naturel de la Terre, illuminée en partie par le soleil, et influençant les marées terrestres.


In [8]:
client = OpenAI()

response = client.embeddings.create(
    input="Your text string goes here", model="text-embedding-3-small"
)

print(response.data[0].embedding)

[0.005115509033203125, 0.017242431640625, -0.018768310546875, -0.0185699462890625, -0.047271728515625, -0.030303955078125, 0.027679443359375, 0.0036468505859375, 0.01125335693359375, 0.006404876708984375, -0.001678466796875, 0.0158233642578125, -0.001255035400390625, -0.007843017578125, 0.0599365234375, 0.050323486328125, -0.0275421142578125, 0.00994110107421875, -0.04034423828125, 0.050018310546875, -0.00043082237243652344, 0.030181884765625, -0.0137176513671875, 0.032958984375, 0.0173187255859375, 0.0167999267578125, -0.0017566680908203125, 0.0204620361328125, 0.040802001953125, -0.0377197265625, -0.026153564453125, -0.050048828125, 0.024200439453125, -0.05523681640625, -0.0322265625, 0.0423583984375, 0.064697265625, 0.01473236083984375, -0.01568603515625, -0.041351318359375, 0.02215576171875, 0.007411956787109375, 0.044952392578125, 0.0070953369140625, -0.0240936279296875, 0.05242919921875, -0.0200958251953125, -0.0322265625, 0.0162811279296875, 0.046478271484375, 0.0238494873046875

In [1]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [2]:
test_message = AIMessage(content="Bonjour")

In [3]:
test_message

AIMessage(content='Bonjour', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])